# Brain Tumor MRI Detector

Binary CNN classifier (PyTorch) for brain MRI slices: **tumor** vs **healthy**.
Dataset: [Brain MRI Images for Brain Tumor Detection (Kaggle)](https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection) — unzip it so that `brain_tumor_dataset/yes` and `brain_tumor_dataset/no` sit next to this notebook.

Pipeline: load → de-duplicate → stratified train/val/test split → train with augmentation, dropout, weight decay and early stopping → evaluate **once** on the untouched test set → run overfitting checks.

> Research/learning project, not a medical device.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tumor_detector as td

cfg = td.Config()              # edit hyper-parameters / data_dir here
td.seed_everything(cfg.seed)
device = td.get_device()
print("device:", device)

## 1. Load data

In [ ]:
images, labels, n_dupes = td.load_images(cfg.data_dir, cfg.img_size)
print(f"{len(labels)} images | tumor: {int(labels.sum())}  healthy: {int((1 - labels).sum())} | exact duplicates removed: {n_dupes}")
td.plot_samples(images, labels);

## 2. Train / validation / test split
Stratified 70/15/15. Normalisation statistics come from the training images only. Augmentation (flip, small rotation/shift/scale) is applied to the training set only.

In [ ]:
train_idx, val_idx, test_idx = td.make_splits(labels, cfg)
for name, idx in (("train", train_idx), ("val", val_idx), ("test", test_idx)):
    print(f"{name:5s}: {len(idx):4d} images, {labels[idx].mean():.1%} tumor")

datasets = td.build_datasets(images, labels, train_idx, val_idx, test_idx)

## 3. Train
The best epoch is chosen on **validation loss**; the test set is not used here.

In [ ]:
model, history = td.train_model(datasets, cfg, device)
td.plot_history(history);

## 4. Test-set evaluation
Run once, after all choices are made. With a small test set the confidence interval matters more than the point estimate.

In [ ]:
res = td.evaluate(model, datasets["test"], device)
lo, hi = res["accuracy_ci95"]
print(f"Test accuracy : {res['accuracy']:.3f}  (95% CI {lo:.3f}-{hi:.3f}, n={res['n']})")
print(f"Precision     : {res['precision']:.3f}")
print(f"Recall        : {res['recall']:.3f}   <- share of real tumors caught")
print(f"F1            : {res['f1']:.3f}")
print(f"ROC AUC       : {res['roc_auc']:.3f}")
td.plot_confusion_matrix(res["confusion_matrix"], "Test confusion matrix");

## 5. Overfitting checks

**5a. Generalisation gap.** Train accuracy (no augmentation) vs validation vs test.

In [ ]:
gap = td.generalisation_gap(model, datasets, device)
for k, v in gap.items():
    print(f"{k:16s} {v:+.3f}" if k == "train_minus_test" else f"{k:16s} {v:.3f}")
print("\n" + td.verdict(gap["train_minus_test"]))

**5b. Stratified 5-fold cross-validation.** Same recipe on 5 different splits of the non-test data. A high spread between folds means a single test number is unreliable.

In [ ]:
cv = td.cross_validate(images, labels, cfg, device, holdout_idx=test_idx, n_splits=5)
acc = np.array([r["test_acc"] for r in cv])
print(f"\nCV held-out accuracy: {acc.mean():.3f} +/- {acc.std():.3f}")

**5c. Shuffled-label test.** Train on randomly permuted labels without early stopping. If train accuracy on random labels is high, the network *can* memorise this dataset, so a high train accuracy on real labels proves nothing. Held-out accuracy should stay near the majority-class rate.

In [ ]:
sh = td.shuffled_label_test(images, labels, train_idx, val_idx, test_idx, cfg, device)
for k, v in sh.items():
    print(f"{k:28s} {v:.3f}")

## 6. Feature maps (optional)

In [ ]:
sample, _ = datasets["test"][0]
td.plot_feature_maps(model, sample, device);

## 7. Save the model

In [ ]:
import torch
torch.save(model.state_dict(), "tumor_cnn.pt")